In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression 
from sklearn.model_selection import GridSearchCV
import data_engineering as de
import data_engineering_roberta as de_rob
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.preprocessing import MinMaxScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score
import simulation

## Data Wrangling

1. Load the Data
2. Create related features (e.g. article counts, open price rolling averages)
3. split by stock

In [ ]:
df = pd.read_csv('../data/complete_next_open.csv')
df.info()

In [ ]:
def overall_sentiment(x:int):
    threshold = .1
    if x > threshold:
        return 'pos'
    elif x < -threshold:
        return 'neg'
    else:
        return 'neu'

In [ ]:
df['overall_sen'] = df['finvader_tot'].apply(overall_sentiment)
df['overall_sen'] = df['overall_sen'].astype('category')
df.info()

In [ ]:
counts = df.groupby(['Market Date', 'Ticker'])['overall_sen'].value_counts()
counts.loc['2019-03-15', 'AAPL']['pos']


In [ ]:
features = ['finvader_neg',
            'finvader_neu',
            'finvader_pos',
            'finvader_tot',
            'Open',
            'High',
            'Low',
            'Close',
            'Volume',
            'Dividends',
            'Stock Splits']
df_mean = df.groupby(['Market Date', 'Ticker'])[features].mean().reset_index()
df_mean

In [ ]:
labels = {'pos_art_count':'pos', 'neg_art_count':'neg', 'neu_art_count':'neu'}
for l in labels:
    df_mean[l] = df_mean.apply(lambda x: counts.loc[x['Market Date'], x['Ticker']][labels[l]], axis = 1)
df_mean.loc[df_mean['finvader_tot'].isna(), 'neu_art_count'] = 0
df_mean['total_articles'] = df_mean['pos_art_count'] + df_mean['neg_art_count'] + df_mean['neu_art_count']


In [ ]:
df_mean['Market Date'] = pd.to_datetime(df_mean['Market Date'])
df_mean.info()

In [ ]:
tickers = df_mean['Ticker'].unique()
ticker_frames = {}
for tick in tickers:
    ticker_frames[tick] = df_mean[df_mean['Ticker'] == tick].set_index('Market Date').drop(columns  = ['Ticker', 'Dividends'])
ticker_frames['AAPL']

In [ ]:
for tick, frame in ticker_frames.items():
    frame['3avg Open'] = frame['Open'].rolling(window = 3).mean()
    frame['7avg Open'] = frame['Open'].rolling(window= 7).mean()
ticker_frames['AAPL']

In [ ]:
for tick, frame in ticker_frames.items():
    frame['indicator'] = -frame['Open'] + frame.shift(-1)['Open']
    frame['indicator'] = frame['indicator'].apply(lambda x: 1 if x >= 0 else 0)
    # ticker_frames[tick] = frame[frame['finvader_tot'].notna()]
    c0 = frame.index.to_series().between(left = '2019-03-15', right = '2024-03-18', inclusive = 'both')
    ticker_frames[tick] = frame[c0]
    ticker_frames[tick] = ticker_frames[tick].fillna(0)
ticker_frames['JNJ']

## Logistic Regression

Time to model.

Will test this model on the simulation at the end

In [ ]:
df_rob = pd.read_csv('../data/complete_next_open_frob_best.csv')

In [ ]:
lr = LogisticRegression(penalty = 'l1', solver = 'liblinear', max_iter=1000)
dummy = DummyClassifier(strategy= 'most_frequent')
parameters = {'C' : [.001, .01, .1, 1, 10, 100]}
clf = GridSearchCV(lr, parameters)
 

In [ ]:
best_para = {}
lr_scores = {}
feature_ranks = {}
dummy_scores = {}
for tick, frame in ticker_frames.items():
    test, train = de.train_test_split(frame)
    X_train  =train.drop(columns = 'indicator')
    y_train =  train['indicator']
    X_test = test.drop(columns = 'indicator')
    y_test = test['indicator'] 
    
    scaler = MinMaxScaler()
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = clf.fit(X_train_scaled, y_train)
    rfe = RFE(estimator=lr, n_features_to_select=5).fit(X_train_scaled, y_train)
    dumb = dummy.fit(X_train_scaled, y_train)

    feature_ranks[tick] = [frame.columns[i] for i in rfe.get_support(1)]
    best_para[tick] = list(model.best_params_.values())

    predict_true = model.predict(X_test_scaled)
    predict_dummy = dummy.predict(X_test_scaled)

    lr_scores[tick] = (accuracy_score(y_test, predict_true), precision_score(y_test, predict_true, zero_division=0), 
                       recall_score(y_test, predict_true), f1_score(y_test, predict_true))
    dummy_scores[tick] = (accuracy_score(y_test, predict_dummy), precision_score(y_test, predict_dummy, zero_division=0),
                           recall_score(y_test, predict_dummy), f1_score(y_test, predict_dummy))




In [ ]:
import numpy as np

# Print individual stock metrics
print(f"{'Ticker':<10} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1':<10}")
print("="*50)

for tick in lr_scores:
    a, b, c, d = lr_scores[tick]
    w, x, y, z = dummy_scores[tick]
    print(f"{tick:<10} {a:<10.4f} {b:<10.4f} {c:<10.4f} {d:<10.4f}")

print("\nAverage Metrics:")
avg_accuracy = np.mean([a for a, _, _, _ in lr_scores.values()])
avg_precision = np.mean([b for _, b, _, _ in lr_scores.values()])
avg_recall = np.mean([c for _, _, c, _ in lr_scores.values()])
avg_f1 = np.mean([d for _, _, _, d in lr_scores.values()])
print(f"{'Average':<10} {avg_accuracy:<10.4f} {avg_precision:<10.4f} {avg_recall:<10.4f} {avg_f1:<10.4f}")

# Print differences from dummy classifier
print("\nDifferences from Dummy Classifier:")
print(f"{'Ticker':<10} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1':<10}")
print("="*50)

for tick in lr_scores:
    a, b, c, d = lr_scores[tick]
    w, x, y, z = dummy_scores[tick]
    print(f"{tick:<10} {a-w:<10.4f} {b-x:<10.4f} {c-y:<10.4f} {d-z:<10.4f}")

print("\nAverage Differences:")
avg_diff_accuracy = np.mean([a-w for (a, _, _, _), (w, _, _, _) in zip(lr_scores.values(), dummy_scores.values())])
avg_diff_precision = np.mean([b-x for (_, b, _, _), (_, x, _, _) in zip(lr_scores.values(), dummy_scores.values())])
avg_diff_recall = np.mean([c-y for (_, _, c, _), (_, _, y, _) in zip(lr_scores.values(), dummy_scores.values())])
avg_diff_f1 = np.mean([d-z for (_, _, _, d), (_, _, _, z) in zip(lr_scores.values(), dummy_scores.values())])
print(f"{'Average':<10} {avg_diff_accuracy:<10.4f} {avg_diff_precision:<10.4f} {avg_diff_recall:<10.4f} {avg_diff_f1:<10.4f}")

In [ ]:
'''
for tick in lr_scores:
    a,b,c,d = lr_scores[tick]
    w, x, y, z = dummy_scores[tick]
    print(tick, a-w, b-x, c-y, d-z)
print('ticker,      accuracy,       precision,      recall,     f1')'
'''

In [ ]:
#Note that the features are just top 5 most important, order is not ranking among the top 5
best_para, feature_ranks

In [ ]:
log_best_param = pd.DataFrame(best_para, index = ['C'])
log_features_top5= pd.DataFrame(feature_ranks)
log_test_scores = pd.DataFrame(lr_scores, index = ['accuracy', 'precision', 'recall', 'f1'])
dummy_test_scores = pd.DataFrame(dummy_scores, index = ['accuracy', 'precision', 'recall', 'f1'])
log_test_scores


## Running Simulation

In [ ]:
cv_trades = [{},{},{},{}]
cv_opens = [{},{},{},{}]
dumb_trades = [{},{},{},{}]

for tick, frame in ticker_frames.items():
    train, test = de.train_test_split(frame)
    #CrossValue
    i=0
    for train_idx, test_idx in de.get_cv_splits(train):
        cv_opens[i][tick] = train.loc[test_idx, "Open"].to_numpy()
        df_tt = train.loc[train_idx]
        df_ho = train.loc[test_idx]
        #df_tt is new train test, df_ho is test set in my cv split. now train model on df_tt
        X_train, y_train = df_tt.drop(columns = 'indicator'), df_tt['indicator']
        X_test, y_test = df_ho.drop(columns = 'indicator'), df_ho['indicator']
        scaler = MinMaxScaler()
        scaler.fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print(y_train)
        #model = clf.fit(X_train_scaled, y_train)
        #dumb = dummy.fit(X_train_scaled, y_train)
        #predict = model.predict(X_test_scaled)
        #predict[predict == 0] = -1
        #pred_dumb = dummy.predict(X_test_scaled)
        #pred_dumb[pred_dumb ==0] == -1
        #cv_trades[i][tick] = predict
        #dumb_trades[i][tick] = pred_dumb
        #i+=1

In [ ]:
def get_performance(trade_dict, test_dict):
    n = len(trade_dict["AAPL"])
    x_t = [1] * n
    for i in range(1,n):
        x_t[i] = x_t[i-1] / 2
        for tick in trade_dict:
            x_t[i] += (x_t[i-1] / 30) * (1 + trade_dict[tick][i-1] * (test_dict[tick][i] - test_dict[tick][i-1]) / test_dict[tick][i-1])
    
    initial_value = x_t[0]
    final_value = x_t[-1]
    growth_percentage = ((final_value - initial_value) / initial_value) * 100
    return growth_percentage #x_t[-1]

In [ ]:
lr_simulation_scores = {}
dumb_simulation_scores = {}
for i in range(4):
    lr_simulation_scores[i] = get_performance(cv_trades[i], cv_opens[i])
    dumb_simulation_scores[i]= get_performance(dumb_trades[i], cv_opens[i])
    print('logistic regression:', lr_simulation_scores[i])
    print('dummy:', dumb_simulation_scores[i])

In [ ]:
log_simulation_scores = pd.DataFrame(lr_simulation_scores, index=['simulation_scores'])
dummy_simulation_scores = pd.DataFrame(dumb_simulation_scores, index=['simulation scores'])
log_simulation_scores

In [ ]:
#dumb_trades

In [ ]:
df_dict = de_rob.separate_by_stock()
df_dict = de_rob.fillna(df_dict)

cv_trades = [{}, {}, {}, {}]
cv_opens = [{}, {}, {}, {}]
dumb_trades = [{},{},{},{}]
y_test_val = [{}, {}, {}, {}]

for tick in df_dict:
    train, test = de_rob.train_test_split(df_dict[tick])

    #features = ["finvader_tot", "pos_art_count", "total_articles", "Open_Diff", "y", "Open"]
    features = ["frob_comp", "pos_art_count", "total_articles", "Open_Diff", "y", "Open"]

    train, test = train[features], test[features]
    train['y'] = train['y'].apply(lambda x: 1 if x >= 0 else 0)
    test['y'] = test['y'].apply(lambda x: 1 if x >= 0 else 0)
    #print(train)
    i = 0
    for train_idx, test_idx in de_rob.get_cv_splits(train):
        cv_opens[i][tick] = train.loc[test_idx, "Open"].to_numpy()
        df_tt = train.loc[train_idx]
        df_ho = train.loc[test_idx]
        X_train, y_train = df_tt.drop(columns=['y']), df_tt['y']
        X_test, y_test = df_ho.drop(columns=['y']), df_ho['y']
        scaler = MinMaxScaler()
        scaler.fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        #print(len(X_train_scaled), len(y_train))
        model = clf.fit(X_train_scaled, y_train)
        dumb = dummy.fit(X_train_scaled, y_train)
        predict = model.predict(X_test_scaled)
        predict[predict == 0] = -1
        pred_dumb = dummy.predict(X_test_scaled)
        pred_dumb[pred_dumb ==0] == -1
        cv_trades[i][tick] = predict
        dumb_trades[i][tick] = pred_dumb
        y_test_val[i][tick] = y_test

        i+=1


In [ ]:
cv_opens[0]

In [ ]:
y_test_val

In [ ]:
# Assuming y_test_val is a list of dictionaries of pandas Series
y_test_val_transformed = []

for y_test_dict in y_test_val:
    transformed_dict = {}
    for tick, series in y_test_dict.items():
        # Remove the index and convert values to 1 and -1
        transformed_dict[tick] = series.reset_index(drop=True).apply(lambda x: 1 if x == 1 else -1).to_numpy()
    y_test_val_transformed.append(transformed_dict)

# Print the transformed y_test_val to verify
#print(y_test_val_transformed)

In [ ]:
y_test_val_transformed

In [ ]:
lr_simulation_scores = {}
dumb_simulation_scores = {}
for i in range(4):
    lr_simulation_scores[i] = get_performance(cv_trades[i], cv_opens[i])
    dumb_simulation_scores[i]= get_performance(dumb_trades[i], cv_opens[i])

mean_lr = np.mean(list(lr_simulation_scores.values()))
mean_dummy = np.mean(list(dumb_simulation_scores.values()))

print('logistic regression:', mean_lr)
print('dummy:', mean_dummy)
    

In [ ]:
cv_trades[0]

In [ ]:
cv_opens[0]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, mean_squared_error
import numpy as np

def get_metrics(trade_dict, test_dict):
    accuracy_list = []
    precision_list = []
    recall_list = []
    rmse_list = []

    for tick in trade_dict:
        y_true = test_dict[tick]
        y_pred = trade_dict[tick]

        # Calculate accuracy
        accuracy = accuracy_score(y_true, y_pred)
        accuracy_list.append(accuracy)

        # Calculate precision
        precision = precision_score(y_true, y_pred, zero_division=0)
        precision_list.append(precision)

        # Calculate recall
        recall = recall_score(y_true, y_pred, zero_division=0)
        recall_list.append(recall)

        # Calculate RMSE
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        rmse_list.append(rmse)

    # Calculate average metrics
    avg_accuracy = np.mean(accuracy_list)
    avg_precision = np.mean(precision_list)
    avg_recall = np.mean(recall_list)
    avg_rmse = np.mean(rmse_list)

    return avg_accuracy, avg_precision, avg_recall, avg_rmse

# Initialize dictionaries to store metrics
accuracy_scores = {}
precision_scores = {}
recall_scores = {}
rmse_scores = {}

# Assuming y_test_val_transformed and cv_trades are defined and contain the data for each fold
for i in range(4):
    accuracy_scores[i], precision_scores[i], recall_scores[i], rmse_scores[i] = get_metrics(y_test_val_transformed[i], cv_trades[i])

# Display the results
#print("Accuracy Scores:", accuracy_scores)
#print("Precision Scores:", precision_scores)
#print("Recall Scores:", recall_scores)
#print("RMSE Scores:", rmse_scores)

# Calculate the mean of the metrics across the 4 folds
mean_accuracy = np.mean(list(accuracy_scores.values()))
mean_precision = np.mean(list(precision_scores.values()))
mean_recall = np.mean(list(recall_scores.values()))
mean_rmse = np.mean(list(rmse_scores.values()))

# Display the results
print("Mean Accuracy:", mean_accuracy)
print("Mean Precision:", mean_precision)
print("Mean Recall:", mean_recall)
print("Mean RMSE:", mean_rmse)

In [ ]:
df_dict = de_rob.separate_by_stock()
df_dict = de_rob.fillna(df_dict)

cv_trades = {}
cv_opens = {}
dumb_trades = {}
y_test_val = {}

for tick in df_dict:
    train, test = de_rob.train_test_split(df_dict[tick])

    #features = ["finvader_tot", "pos_art_count", "total_articles", "Open_Diff", "y", "Open"]
    features = ["frob_comp", "pos_art_count", "total_articles", "Open_Diff", "y", "Open"]

    train, test = train[features], test[features]
    train['y'] = train['y'].apply(lambda x: 1 if x >= 0 else 0)
    test['y'] = test['y'].apply(lambda x: 1 if x >= 0 else 0)

    X_train, y_train = train.drop(columns=['y']), train['y']
    X_test, y_test = test.drop(columns=['y']), test['y']
    #print(train)
    
    scaler = MinMaxScaler()
    scaler.fit(X_train)

    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = clf.fit(X_train_scaled, y_train)
    dumb = dummy.fit(X_train_scaled, y_train)

    predict = model.predict(X_test_scaled)
    predict[predict == 0] = -1

    pred_dumb = dummy.predict(X_test_scaled)
    pred_dumb[pred_dumb ==0] == -1

    cv_trades[tick] = predict
    dumb_trades[tick] = pred_dumb
    y_test_val[tick] = y_test.to_numpy()
    cv_opens[tick] = test['Open'].to_numpy()


In [ ]:
len(cv_trades['AAPL']), len(cv_opens['AAPL']), len(y_test_val['AAPL'])

In [ ]:
def get_performance(trade_dict, test_dict):
    n = len(trade_dict["AAPL"])
    x_t = [1] * n
    for i in range(1,n):
        x_t[i] = x_t[i-1] / 2
        for tick in trade_dict:
            x_t[i] += (x_t[i-1] / 30) * (1 + trade_dict[tick][i-1] * (test_dict[tick][i] - test_dict[tick][i-1]) / test_dict[tick][i-1])
    
    initial_value = x_t[0]
    final_value = x_t[-1]
    growth_percentage = ((final_value - initial_value) / initial_value) * 100
    return growth_percentage #x_t[-1]

In [ ]:
def get_performance_original(trade_dict, test_dict):
    n = len(trade_dict["AAPL"])
    x_t = [1] * n
    for i in range(1 , n):
        x_t[i] = x_t[i-1] / 2
        for tick in trade_dict:
            x_t[i] += (x_t[i-1] / 30) * (1 + trade_dict[tick][i-1] * (test_dict[tick][i] - test_dict[tick][i-1]) / test_dict[tick][i-1])
    return x_t[-1]

In [ ]:
print(get_performance(cv_trades, cv_opens))
print(get_performance(dumb_trades, cv_opens))

In [ ]:
# Assuming y_test_val is a dictionary of numpy arrays
y_test_val_transformed = {}

for tick, array in y_test_val.items():
    # Convert values to 1 and -1
    y_test_val_transformed[tick] = np.where(array == 1, 1, -1)

# Print the transformed y_test_val to verify
#print(y_test_val_transformed)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, mean_squared_error
import numpy as np

def get_metrics(trade_dict, test_dict):
    accuracy_list = []
    precision_list = []
    recall_list = []
    rmse_list = []

    for tick in trade_dict:
        y_true = test_dict[tick]
        y_pred = trade_dict[tick]

        # Calculate accuracy
        accuracy = accuracy_score(y_true, y_pred)
        accuracy_list.append(accuracy)

        # Calculate RMSE
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        rmse_list.append(rmse)

    # Calculate average metrics
    avg_accuracy = np.mean(accuracy_list)
    avg_precision = np.mean(precision_list)
    avg_recall = np.mean(recall_list)
    avg_rmse = np.mean(rmse_list)

    return avg_accuracy, avg_rmse

# Initialize dictionaries to store metrics
accuracy_scores = {}
precision_scores = {}
recall_scores = {}
rmse_scores = {}

# Assuming y_test_val_transformed and cv_trades are defined and contain the data for each fold
accuracy_scores, rmse_scores = get_metrics(y_test_val_transformed, cv_trades)


# Display the results
print("Accuracy:", accuracy_scores)
print("RMSE:", rmse_scores)